In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from core.GenericDataPipeline import GenericDataPipeline
import pandas as pd
import numpy as np
import optuna
import kagglehub
import os
from core.seed_utils import SEED, set_all_seeds

pd.set_option('future.infer_string', False)
set_all_seeds()

All random seeds have been set to: 42


# [Weather Data](https://www.kaggle.com/datasets/rever3nd/weather-data)

In [2]:
pipeline = GenericDataPipeline()

path = kagglehub.dataset_download("rever3nd/weather-data")
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
csv_path = os.path.join(path, csv_files[0])
df = pd.read_csv(csv_path)

columns_to_drop = [
    'Unnamed: 0', 'Date', 'RISK_MM',
    'Humidity3pm', 'Pressure3pm', 'Cloud3pm', 'Temp3pm', 'WindDir3pm', 'WindSpeed3pm'
]
df = df.drop(columns=columns_to_drop)
df = pipeline.preprocessing(df)

label = "RainTomorrow"
print(f"Dataset shape: {df.shape}")
print(f"Target: {label}")
print(f"Target distribution:\n{df[label].value_counts()}")

Dataset shape: (25000, 16)
Target: RainTomorrow
Target distribution:
RainTomorrow
0    19405
1     5595
Name: count, dtype: int64


In [3]:
X = df.drop(label, axis=1)
y = df[label]

feature_scores = pipeline.rank_features(X, y)
features_with_nulls = feature_scores[feature_scores['null_ratio'] > 0]

print(f"\nFeatures with nulls ({len(features_with_nulls)}):")
print(features_with_nulls[['feature_name', 'null_ratio']].to_string(index=False))

Starting feature importance calculation with XGBoost and 5-fold CV...
Mean target value (for base_score): 0.223800
Training XGBoost models across 5 folds...
Unique target values: {np.int8(0), np.int8(1)}
Fold 1/5
  Train positive rate: 0.2238, Validation positive rate: 0.2238


Fold 2/5
  Train positive rate: 0.2238, Validation positive rate: 0.2238


Fold 3/5
  Train positive rate: 0.2238, Validation positive rate: 0.2238


Fold 4/5
  Train positive rate: 0.2238, Validation positive rate: 0.2238


Fold 5/5
  Train positive rate: 0.2238, Validation positive rate: 0.2238


Mean AUC across 5 successful folds: 0.8453

Feature scores:
----------------------------------------------------------------------------------------------------
Rank  | Feature                        | Gain Score      | Null Ratio
----------------------------------------------------------------------------------------------------
15    | RainToday                      | 1.000000 | 0.0112
4     | Rainfall                       | 0.461014 | 0.0112
11    | Humidity9am                    | 0.255011 | 0.0156
6     | Sunshine                       | 0.209953 | 0.7334
13    | Cloud9am                       | 0.204819 | 0.4346
8     | WindGustSpeed                  | 0.185225 | 0.1382
3     | MaxTemp                        | 0.126075 | 0.0070
2     | MinTemp                        | 0.120982 | 0.0132
1     | Location                       | 0.094689 | 0.0000
7     | WindGustDir                    | 0.094371 | 0.1383
12    | Pressure9am                    | 0.090101 | 0.1931
14    | Temp9am    

In [4]:
n_trials = 20

print("Running Optuna optimization...")

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study.optimize(
    lambda trial: pipeline.objective(trial, features_with_nulls, feature_scores, df, label),
    n_trials=n_trials
)

print(f"\nBest trial:")
print(f"Validation loss + penalty: {study.best_value}")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

Running Optuna optimization...
selected_features
['RainToday', 'Rainfall', 'Humidity9am', 'Sunshine', 'Cloud9am', 'WindGustSpeed', 'MaxTemp', 'MinTemp', 'WindGustDir', 'Pressure9am']

=== Breakdown BEFORE splitting ===
has_extended
1    24975
0       25
Name: count, dtype: int64
Extended percentage: 99.9 %
Train set distribution:
RainTomorrow  has_extended
0             0                   4
              1               15520
1             0                  16
              1                4460
dtype: int64

Test set distribution:
RainTomorrow  has_extended
0             0                  1
              1               3880
1             0                  4
              1               1115
dtype: int64

⚠️ One of the train/test groups has fewer than 100 samples!
selected_features
['RainToday', 'Rainfall', 'Humidity9am', 'Sunshine', 'Cloud9am', 'WindGustSpeed', 'MaxTemp', 'MinTemp', 'WindGustDir', 'Pressure9am']

=== Breakdown BEFORE splitting ===
has_extended
1    24992
0      

Train set distribution:
RainTomorrow  has_extended
0             0                  33
              1               15491
1             0                   6
              1                4470
dtype: int64

Test set distribution:
RainTomorrow  has_extended
0             0                  8
              1               3873
1             0                  2
              1               1117
dtype: int64

⚠️ One of the train/test groups has fewer than 100 samples!
selected_features
['RainToday', 'Rainfall', 'Humidity9am', 'Sunshine', 'Cloud9am', 'WindGustSpeed', 'MaxTemp', 'MinTemp', 'WindGustDir', 'Pressure9am']

=== Breakdown BEFORE splitting ===
has_extended
1    24926
0       74
Name: count, dtype: int64


Extended percentage: 99.7 %
Train set distribution:
RainTomorrow  has_extended
0             0                  31
              1               15493
1             0                  28
              1                4448
dtype: int64

Test set distribution:
RainTomorrow  has_extended
0             0                  8
              1               3873
1             0                  7
              1               1112
dtype: int64

⚠️ One of the train/test groups has fewer than 100 samples!
selected_features
['RainToday', 'Rainfall', 'Humidity9am', 'Sunshine', 'Cloud9am', 'WindGustSpeed', 'MaxTemp', 'MinTemp', 'WindGustDir', 'Pressure9am']

=== Breakdown BEFORE splitting ===
has_extended
1    24826
0      174
Name: count, dtype: int64
Extended percentage: 99.3 %
Train set distribution:
RainTomorrow  has_extended
0             0                 100
              1               15424
1             0                  39
              1                4437
dtype: int64

Test set dist

********** run results **********
{'max_depth': 5, 'learning_rate': 0.07969454818643933, 'n_estimators': 759, 'min_child_weight': 5, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'gamma': 0.2904180608409973, 'lambda': 8.661895281603577, 'alpha': 6.011549002420345, 'scale_pos_weight': 5.248435466776273, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cpu'}
*****
[0.8338020618389856, 0.8382209426161464, 0.8432133082259166]
********** run results **********


********** run results **********
{'max_depth': 3, 'learning_rate': 0.08706020878304854, 'n_estimators': 850, 'min_child_weight': 2, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.6733618039413735, 'gamma': 1.5212112147976886, 'lambda': 5.2480395598907466, 'alpha': 4.320018241402516, 'scale_pos_weight': 2.7473748411882513, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cpu'}
*****
[0.8382330331205141, 0.8423123413146737, 0.8450521513098229]
********** run results **********


...

********** run results **********
{'max_depth': 4, 'learning_rate': 0.01811866911332789, 'n_estimators': 516, 'min_child_weight': 4, 'subsample': 0.7638022988255873, 'colsample_bytree': 0.7814229301059501, 'gamma': 1.0263661982169017, 'lambda': 7.809855909854404, 'alpha': 7.678761741780726, 'scale_pos_weight': 6.214116345182809, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cpu'}
*****
[0.8341411162090646, 0.8426528947038077, 0.8474023293563735]
********** run results **********
Best parameters found: {'max_depth': 5, 'learning_rate': 0.04257148895072214, 'n_estimators': 617, 'min_child_weight': 3, 'subsample': 0.6596402006946023, 'colsample_bytree': 0.659941629468189, 'gamma': 1.8968687659549008, 'lambda': 9.692729250635129, 'alpha': 7.616865670368867, 'scale_pos_weight': 2.558943282255626}
Best CV AUC score: 0.8504

===== Detailed Cross-Validation Results with Best Parameters =====
Params: {'max_depth': 5, 'learning_rate': 0.04257148895072214, 'n_estimat

Fold 1 AUC: 0.8460


Fold 2 AUC: 0.8514


Fold 3 AUC: 0.8534

Final CV Results - Mean AUC: 0.8502, Std Dev: 0.0031


Test set AUC (with extended features): 0.8807
Test set AUC (without extended features): 0.8497
Overall test set AUC: 0.8595
RainToday: 0.3160
Rainfall: 0.1066
Humidity9am: 0.0867
Cloud9am: 0.0676
WindGustSpeed: 0.0667
MaxTemp: 0.0382
MinTemp: 0.0377
WindGustDir: 0.0330
Pressure9am: 0.0317
Location: 0.0300
Temp9am: 0.0294
WindDir9am: 0.0253
WindSpeed9am: 0.0244
Evaporation: 0.0225
Sunshine: 0.0584
has_extended: 0.0256

Top 10 features by importance:
RainToday: 0.3160
Rainfall: 0.1066
Humidity9am: 0.0867
Cloud9am: 0.0676
WindGustSpeed: 0.0667
Sunshine: 0.0584
MaxTemp: 0.0382
MinTemp: 0.0377
WindGustDir: 0.0330
Pressure9am: 0.0317
len with ext 1333
len without ext 3667

Base model (no extended)
AUC: 0.8338, Accuracy: 0.7947, F1 Score: 0.5837

Extended model (with extended)
AUC: 0.8451, Accuracy: 0.7644, F1 Score: 0.6075

Combined model (no extended)
AUC: 0.8321, Accuracy: 0.7922, F1 Score: 0.5809

Combined model (with extended)
AUC: 0.8600, Accuracy: 0.8072, F1 Score: 0.6416

Results Summ

# Final Results